# Lab 7: Visual SLAM with MapAnything

In this lab we will build a 3D map of our drone flight space from **one photo**. We will use [MapAnything](https://github.com/facebookresearch/map-anything), a large transformer model from Meta and CMU that takes in images and directly predicts 3D geometry: per-pixel depth, camera intrinsics, and a 3D point for every pixel.

A neural network that estimates depth from a single image is solving a problem that is mathematically ill-posed (a small object nearby looks identical to a large object far away). In this lab, we will see how well this estimation works in practice. We will use our knowledge of two things:
1. **Your camera.** Your phone writes its focal length into the image file (EXIF metadata). From this we can build the intrinsic matrix $K$ ourselves.
2. **The lab space.** You can measure it with a tape measure.

### Before you start: take your photo
1. Stand inside the netted flight area and take a **landscape** photo of the flight space with an iPhone (or any phone that records 35 mm-equivalent focal length in EXIF) using the **main 1x lens**. Do not use Portrait mode, the 0.5x ultra-wide, digital zoom, or a cropped/edited photo. Make sure that the photo is taken from a point of view where you can see objects in the background (e.g., chairs, tables, etc.); including these familiar objects in the picture will improve depth estimation, as discussed in class.
2. Get the original file onto your computer **with metadata intact** (AirDrop, USB, or "Export Unmodified Original" from Photos). Screenshots and photos sent through messaging apps usually lose their EXIF data. HEIC and JPEG files both work.
3. While you are there, **measure with a tape measure** (in meters):
    - the height of the phone's camera above the floor when you took the photo,
    - the distance between the centers of the bases of two foam-covered obstacle poles that are both visible in your photo,
    - the length of one obstacle foam tube.

### Setup
MapAnything needs Python $\geq$ 3.10, so it does **not** run in the `mae345` environment. Either:
- **[Google Colab](https://colab.research.google.com/) (recommended):** Runtime → Change runtime type → GPU. Upload this notebook and your photo, then uncomment and run the `pip install` cell below.
- **Locally:** `conda create -n mapanything python=3.12 -y`, `conda activate mapanything`, then `pip install git+https://github.com/facebookresearch/map-anything.git matplotlib jupyter`. Inference on a CPU works for a single image, but it will take a minute or two.

The first time you load the model it downloads about 5 GB of weights from Hugging Face.

In [ ]:
# Uncomment and run once if you are on Google Colab (takes a few minutes)
# !pip install -q git+https://github.com/facebookresearch/map-anything.git

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps
from PIL.ExifTags import TAGS
from matplotlib.colors import rgb_to_hsv

import pillow_heif
pillow_heif.register_heif_opener()   # lets PIL open iPhone .HEIC files

In [ ]:
# TODO: point this at your own photo and fill in your tape-measure numbers (meters)
IMAGE_PATH = "IMG_5495.jpg" # You can upload your photo to Google Colab by clicking on "Files" in the left tab

CAMERA_HEIGHT_MEASURED = None   # height of phone camera above the floor [m]
POLE_SPACING_MEASURED  = None   # distance between the bases of two poles [m]
FOAM_LENGTH_MEASURED   = None   # length of one obstacle foam tube [m]

# Part I: Camera Intrinsics from EXIF (10 pts)

### The pinhole camera model
We will use a pinhole camera model. For this lab, we will use the OpenCV camera convention (+X right, +Y down, +Z forward out of the lens) since this is what MapAnything uses. A 3D point $(X, Y, Z)$ in the camera frame projects to pixel $(u, v)$:

$$u = f_x \frac{X}{Z} + c_x, \qquad v = f_y \frac{Y}{Z} + c_y, \qquad\text{or}\qquad Z\begin{bmatrix}u\\v\\1\end{bmatrix} = \underbrace{\begin{bmatrix} f_x & 0 & c_x\\ 0 & f_y & c_y \\ 0 & 0 & 1\end{bmatrix}}_{K}\begin{bmatrix}X\\Y\\Z\end{bmatrix}.$$

Here, $u$ is the column index and $v$ is the row index; $f_x, f_y$ are focal lengths **in pixels** and $(c_x, c_y)$ is the principal point in pixels (i.e., the center point in the image). 

### Reading the focal length from EXIF
The phone records two focal lengths in the image metadata:
- `FocalLength`: the *physical* focal length of the lens in mm (5.96 mm on an iPhone 16). To convert it to pixels we would need the physical size of a pixel on the sensor, which is not in the file.
- `FocalLengthIn35mmFilm`: the focal length a lens would need on a standard 36 mm $\times$ 24 mm "full-frame" sensor to give the **same field of view** (26 mm on an iPhone 16). This one is useful, because field of view is something we can relate to pixels.

The 35 mm-equivalent focal length is defined using the sensor **diagonal**: two cameras have the same field of view when focal length / sensor diagonal is the same. The full-frame diagonal is $\sqrt{36^2 + 24^2} \approx 43.27$ mm. Because the ratio is unitless, it also holds when the "sensor" is measured in pixels:

$$\frac{f_{px}}{\sqrt{W^2 + H^2}} = \frac{f_{35}}{43.27\text{ mm}},$$

where $W \times H$ is the image size in pixels.

The cell below reads the EXIF tags and the image size for you.

In [ ]:
# Do not modify
def read_exif(path):
    img = Image.open(path)
    exif = img.getexif()
    tags = {TAGS.get(k, k): v for k, v in exif.items()}
    tags.update({TAGS.get(k, k): v for k, v in exif.get_ifd(0x8769).items()})  # camera settings sub-block
    img = ImageOps.exif_transpose(img).convert("RGB")   # apply the phone's rotation flag
    return img, tags

img_full, exif = read_exif(IMAGE_PATH)
W, H = img_full.size
for key in ["Make", "Model", "LensModel", "FocalLength", "FocalLengthIn35mmFilm"]:
    print(f"{key:>22}: {exif.get(key)}")
print(f"{'Image size (W x H)':>22}: {W} x {H}")

assert exif.get("FocalLengthIn35mmFilm"), "No 35mm-equivalent focal length found; is the EXIF metadata intact?"

plt.figure(figsize=(8, 6)); plt.imshow(img_full); plt.axis("off"); plt.show()

### Camera intrinsics
Compute the focal length in pixels and build the $3\times 3$ intrinsic matrix `K_full` for the full-resolution image. Assume:
- square pixels ($f_x = f_y$),
- the principal point is at the center of the image: $c_x = W/2$, $c_y = H/2$.

Also compute the horizontal field of view in degrees. (Hint: draw the triangle from the pinhole to the left and right edges of the image, similar to the theory problem in this assignment.)

In [ ]:
f35 = float(exif["FocalLengthIn35mmFilm"])   # [mm]

# TODO: focal length in pixels, intrinsic matrix, and horizontal field of view
f_px = None
K_full = None        # 3x3 numpy array
hfov_deg = None

print("f_px =", f_px)
print("K_full =\n", K_full)
print("Horizontal FOV [deg] =", hfov_deg)

### Intrinsics at the network's resolution
MapAnything does not run on a 24-megapixel image. Its preprocessing step (`preprocess_inputs`) shrinks the image to $518 \times 392$ pixels for a 4:3 photo:
1. **Resize** the image by a uniform scale factor $s = \max(518/W,\; 392/H)$, so that the resized image covers the target size, giving a $W_r \times H_r$ image with $W_r = \lfloor sW \rfloor$, $H_r = \lfloor sH \rfloor$.
2. **Center-crop** to exactly $518 \times 392$, removing `left` $= \lfloor (W_r - 518)/2 \rfloor$ columns from the left and `top` $= \lfloor (H_r - 392)/2 \rfloor$ rows from the top.

All of MapAnything's outputs (depth, points, intrinsics) live at this $518 \times 392$ resolution, so we need $K$ for **this** image.

Work out how $f_x, f_y, c_x, c_y$ change when (1) the image is scaled by $s$, and (2) the image is cropped. Then fill in `K_net` below.

In [ ]:
W_net, H_net = 518, 392
s = max(W_net / W, H_net / H)
W_r, H_r = int(np.floor(s * W)), int(np.floor(s * H))
left, top = (W_r - W_net) // 2, (H_r - H_net) // 2
print(f"scale s = {s:.5f}, resized to {W_r} x {H_r}, crop left = {left}, top = {top}")

# TODO: intrinsics of the 518 x 392 network image
K_net = None
print("K_net =\n", K_net)

The cell below checks your answer against MapAnything's own preprocessing code, which transforms $K$ along with the image. A difference of under a pixel is fine: the library uses a convention where pixel centers sit at half-integer coordinates, which shifts $c_x, c_y$ slightly.

In [ ]:
# Do not modify
from mapanything.utils.image import preprocess_inputs

views_check = preprocess_inputs([{"img": img_full, "intrinsics": K_full.astype(np.float32)}])
K_lib = views_check[0]["intrinsics"][0].numpy()
print("MapAnything preprocessing K =\n", K_lib)
print("Image tensor shape:", tuple(views_check[0]["img"].shape))
assert np.allclose(K_net, K_lib, atol=1.0), "K_net does not match MapAnything's preprocessing"
print("Looks good!")

# Part II: Running MapAnything (5 pts)

We first run MapAnything with **only the image** as input. In this mode the network must also guess the camera intrinsics: it predicts a unit ray direction for every pixel, and fits a pinhole $K$ to those rays.

The relevant outputs (each has a leading batch dimension of 1, which we strip off) are:

| key | shape | meaning |
|---|---|---|
| `depth_z` | $(H, W, 1)$ | $Z$-depth of each pixel in the camera frame [m] |
| `pts3d_cam` | $(H, W, 3)$ | 3D point of each pixel in the camera frame [m] |
| `intrinsics` | $(3, 3)$ | pinhole $K$ fitted to the predicted rays |
| `conf` | $(H, W)$ | per-pixel confidence (higher is better) |
| `mask` | $(H, W, 1)$ | pixels the model considers valid (sky, depth edges, etc. removed) |

Run the cells below. You do not need to modify them.

In [ ]:
# Do not modify
from mapanything.models import MapAnything

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Downloads the weights the first time. "facebook/map-anything-apache" is an Apache-2.0 licensed alternative.
model = MapAnything.from_pretrained("facebook/map-anything").to(device)
model.eval();

In [ ]:
# Do not modify
def run_mapanything(view):
    views = preprocess_inputs([view])
    with torch.no_grad():
        pred = model.infer(views, memory_efficient_inference=False,
                           use_amp=(device == "cuda"), amp_dtype="bf16",
                           apply_mask=True, mask_edges=True)[0]
    to_np = lambda key: pred[key][0].float().cpu().numpy()
    return {
        "img":   to_np("img_no_norm"),          # (H, W, 3) RGB in [0, 1]
        "depth": to_np("depth_z")[..., 0],      # (H, W)
        "pts":   to_np("pts3d_cam"),            # (H, W, 3)
        "K":     to_np("intrinsics"),           # (3, 3)
        "conf":  to_np("conf"),                 # (H, W)
        "mask":  to_np("mask")[..., 0] > 0.5,   # (H, W) boolean
    }

out_ma = run_mapanything({"img": img_full})     # image only: MapAnything guesses the intrinsics

img_net  = out_ma["img"]
depth_ma = out_ma["depth"]
pts_ma   = out_ma["pts"]
K_ma     = out_ma["K"]
valid    = out_ma["mask"] & (depth_ma > 0)

print("K predicted by MapAnything =\n", K_ma)

In [ ]:
# Do not modify
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].imshow(img_net); ax[0].set_title("Network input (518 x 392)")
im = ax[1].imshow(np.where(valid, depth_ma, np.nan), cmap="turbo"); ax[1].set_title("MapAnything depth $Z$ [m]")
plt.colorbar(im, ax=ax[1])
im = ax[2].imshow(np.log(out_ma["conf"]), cmap="magma"); ax[2].set_title("log confidence")
plt.colorbar(im, ax=ax[2])
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

**Question (5 pts):** Look at the depth and confidence maps. Which regions of the scene does MapAnything find hardest (low confidence or masked out)? Does this match with your intuition?

*Your answer here.*

# Part III: From Depth Map to Point Map (15 pts)

### Back-projection 
A depth map stores only $Z$ for every pixel. To recover the full 3D point, we invert the pinhole projection:

$$X = \frac{(u - c_x)\, Z}{f_x}, \qquad Y = \frac{(v - c_y)\, Z}{f_y}, \qquad Z = Z.$$

Implement `backproject(depth, K)`, which takes an $(H, W)$ depth map and returns an $(H, W, 3)$ **point map** whose entry `[v, u]` is the 3D point $(X, Y, Z)$ seen by pixel $(u, v)$. Use integer pixel indices: $u \in \{0, \dots, W-1\}$ is the column, $v \in \{0, \dots, H-1\}$ is the row (this matches MapAnything's convention).

Do this **without for loops**. `np.meshgrid(np.arange(W), np.arange(H))` returns two $(H, W)$ arrays holding the $u$ and $v$ coordinate of every pixel, and `np.stack(..., axis=-1)` stacks three $(H, W)$ arrays into an $(H, W, 3)$ array.

In [ ]:
# TODO
def backproject(depth, K):
    '''
    depth: (H, W) array of Z-depths
    K:     (3, 3) intrinsic matrix
    returns: (H, W, 3) array of 3D points in the camera frame
    '''
    pass

The next cell tests your function in two ways:
1. **Round trip:** random 3D points are projected into a small image with a known $K$, then back-projected. You should get the same points back.
2. **Consistency with MapAnything:** back-projecting MapAnything's depth with MapAnything's **own** $K$ should approximately reproduce its `pts3d_cam`. It will not be exact: the network predicts a separate ray for each pixel, and `K_ma` is only the best pinhole fit to those rays.

In [ ]:
# Do not modify
# 1. Round trip on a toy camera
rng = np.random.default_rng(0)
K_toy = np.array([[50.0, 0, 20.0], [0, 55.0, 15.0], [0, 0, 1]])
Z_toy = rng.uniform(1, 5, size=(30, 40))
u_toy, v_toy = np.meshgrid(np.arange(40), np.arange(30))
P_true = np.stack([(u_toy - 20.0) * Z_toy / 50.0, (v_toy - 15.0) * Z_toy / 55.0, Z_toy], axis=-1)
assert np.allclose(backproject(Z_toy, K_toy), P_true), "Round-trip test failed"
print("Round-trip test passed.")

# 2. Consistency with MapAnything's own point map
P_check = backproject(depth_ma, K_ma)
err_check = np.linalg.norm(P_check - pts_ma, axis=-1)[valid]
rel_check = err_check / depth_ma[valid]
print(f"Median distance between backproject(depth_ma, K_ma) and pts3d_cam: {np.median(err_check)*100:.2f} cm "
      f"({np.median(rel_check)*100:.2f}% of depth)")
assert np.median(rel_check) < 0.02, "Large mismatch: check your u/v and cx/cy conventions"

### Your point map
Now we will build the point map that uses MapAnything's depth but **your** EXIF-based intrinsics `K_net`. Call it `pts_exif`. Then we will compare the two sets of intrinsics: the percent difference in focal length and the horizontal field of view implied by each $K$ (at the $518 \times 392$ resolution).

In [ ]:
# Do not modify
pts_exif = backproject(depth_ma, K_net)

f_exif, f_ma = K_net[0, 0], K_ma[0, 0]
focal_diff_pct = 100 * (f_ma - f_exif) / f_exif
hfov_exif_deg = np.degrees(2 * np.arctan(W_net / (2 * f_exif)))
hfov_ma_deg = np.degrees(2 * np.arctan(W_net / (2 * f_ma)))

print(f"focal length  EXIF: {f_exif:.1f} px   MapAnything: {f_ma:.1f} px   difference: {focal_diff_pct:+.1f}%")
print(f"horizontal FOV EXIF: {hfov_exif_deg:.1f} deg  MapAnything: {hfov_ma_deg:.1f} deg")
print(f"principal point EXIF: ({K_net[0,2]:.1f}, {K_net[1,2]:.1f})  MapAnything: ({K_ma[0,2]:.1f}, {K_ma[1,2]:.1f})")

# Part IV: Comparing the Two Point Maps

We now have two point maps built from the **same depth map** with two different intrinsic matrices:
- `pts_ma`: MapAnything's own `pts3d_cam` (its predicted rays),
- `pts_exif`: the same depth, back-projected with the EXIF intrinsics.

### Error map
We will compute, for every valid pixel, the Euclidean distance between the two 3D points, `err` (an $(H, W)$ array, set to `np.nan` at invalid pixels). Also compute the **relative** error `rel_err = err / depth_ma`. Then we print the median and 90th percentile of each (use `np.nanmedian`, `np.nanpercentile`).

In [ ]:
err = np.where(valid, np.linalg.norm(pts_exif - pts_ma, axis=-1), np.nan)
rel_err = err / depth_ma

print(f"3D distance:    median {np.nanmedian(err)*100:.1f} cm, 90th pct {np.nanpercentile(err, 90)*100:.1f} cm")
print(f"relative error: median {np.nanmedian(rel_err)*100:.2f} %,  90th pct {np.nanpercentile(rel_err, 90)*100:.2f} %")

In [ ]:
# Do not modify
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
im = ax[0].imshow(err * 100, cmap="viridis"); ax[0].set_title("|pts_exif - pts_ma| [cm]"); plt.colorbar(im, ax=ax[0])
im = ax[1].imshow(rel_err * 100, cmap="viridis"); ax[1].set_title("relative error [% of depth]"); plt.colorbar(im, ax=ax[1])
for a in ax:
    a.plot(K_net[0, 2], K_net[1, 2], "r+", ms=15, mew=2)   # principal point
    a.axis("off")
plt.tight_layout(); plt.show()

### Visualizing the point map
Run the cell below to view the EXIF point map in 3D, colored by the image. (If you installed `ipympl`, put `%matplotlib widget` at the top of the cell to rotate it with the mouse.)

In [ ]:
# Do not modify
fig = plt.figure(figsize=(8, 6))
ax3 = fig.add_subplot(projection="3d")
sub = np.zeros_like(valid); sub[::3, ::3] = True; sub &= valid
P = pts_exif[sub]
ax3.scatter(P[:, 0], P[:, 2], -P[:, 1], c=img_net[sub], s=1)
ax3.set(xlabel="X [m]", ylabel="Z [m]", zlabel="-Y [m]", title="Point map (EXIF K)")
ax3.view_init(elev=20, azim=-70)
plt.tight_layout(); plt.show()

# Part V: Ground Truth, and Does Calibration Help? (30 pts)

### Giving MapAnything the intrinsics
MapAnything can take extra inputs besides the image. Below we run it a second time, now also passing in `K_full`. The network then uses our rays instead of guessing them, and can use the known field of view when estimating depth. (`preprocess_inputs` rescales and crops `K_full` exactly as you did in Part I.)

In [ ]:
# Do not modify
out_cal = run_mapanything({"img": img_full, "intrinsics": K_full.astype(np.float32)})
depth_cal, pts_cal = out_cal["depth"], out_cal["pts"]
valid_cal = valid & out_cal["mask"] & (depth_cal > 0)
print("K returned by calibrated run =\n", out_cal["K"])

### How much did the depth change?
Compute the per-pixel relative depth difference $|Z_{cal} - Z_{ma}| / Z_{ma}$ over `valid_cal` pixels and print its median. Also print the median of the ratio $Z_{cal}/Z_{ma}$: is the calibrated depth systematically larger or smaller?

In [ ]:
# TODO

### Fitting the floor plane 
The artificial turf gives us a large, flat, easy-to-find surface. The cell below selects turf pixels by color (green hue, fairly saturated, not too bright, in the lower part of the image). Check that the mask looks right for your photo and adjust the thresholds if needed.

A plane is the set of points $\{P : n^\top P + d = 0\}$ with unit normal $\|n\| = 1$. Given $N$ points $P_i$, the least-squares plane passes through the centroid $\bar P$, and its normal $n$ is the direction in which the centered points vary **least**: the right singular vector of the $N \times 3$ matrix of centered points that corresponds to the **smallest** singular value. Then $d = -n^\top \bar P$.

We provide an implementation of `fit_plane(P)` below. Since the camera sits at the origin, the camera's height above the floor is simply $|d|$, the distance from the origin to the plane.

In [ ]:
# Do not modify (but feel free to tweak the thresholds for your photo)
hsv = rgb_to_hsv(np.clip(img_net, 0, 1))
rows = np.arange(H_net)[:, None] * np.ones((1, W_net))
turf = (hsv[..., 0] > 0.22) & (hsv[..., 0] < 0.45) & (hsv[..., 1] > 0.2) & (hsv[..., 2] < 0.6) & (rows > 0.6 * H_net)
turf &= valid_cal

plt.figure(figsize=(6, 4.5)); plt.imshow(img_net); plt.imshow(np.ma.masked_where(~turf, turf), cmap="autumn", alpha=0.7)
plt.title(f"Turf mask ({turf.sum()} pixels)"); plt.axis("off"); plt.show()

In [1]:
# Do not modify
def fit_plane(P):
    '''
    P: (N, 3) array of points
    returns: n (3,) unit normal, d (scalar), such that n @ p + d = 0 on the plane
    '''
    centroid = P.mean(axis=0)
    _, _, Vt = np.linalg.svd(P - centroid, full_matrices=False)
    n = Vt[-1]
    d = -n @ centroid
    return n, d

The cell below fits the floor in each of the three point maps. It refits a few times, each time keeping only points within 5 cm of the current plane, so that stray netting pixels do not pull the fit. It also reports the camera's pitch, i.e., the angle between the optical axis ($+Z$) and the floor.

In [ ]:
# Do not modify
def robust_floor(P_map, iters=3, thresh=0.05):
    P = P_map[turf]
    keep = np.ones(len(P), dtype=bool)
    for _ in range(iters):
        n, d = fit_plane(P[keep])
        keep = np.abs(P @ n + d) < thresh
    if n[1] > 0:  # make the normal point "up" (camera -Y)
        n, d = -n, -d
    return n, d, keep.mean()

floor = {}
for name, P_map in [("MapAnything K", pts_ma), ("EXIF K", pts_exif), ("Calibrated run", pts_cal)]:
    n, d, inlier_frac = robust_floor(P_map)
    pitch = np.degrees(np.arcsin(np.clip(-n[2], -1, 1)))
    floor[name] = (n, d)
    print(f"{name:>15}: camera height = {abs(d):.3f} m, pitch = {pitch:5.1f} deg down, inliers = {inlier_frac*100:.0f}%")
print(f"{'Tape measure':>15}: camera height = {CAMERA_HEIGHT_MEASURED} m")

### An obstacle map for the drone
For a drone, the most useful view of the room is often from **above** (as in our RRT Lab): where are the obstacles on the floor? With the floor plane in hand, we can express every point in a floor-aligned frame: height above the floor $h = n^\top P + d$, and two horizontal coordinates obtained by projecting the camera's right ($+X$) and forward ($+Z$) axes onto the floor. Keeping only points between 0.3 m and 2 m above the floor, the foam poles show up as small clusters. The cell below does this for both the MapAnything-$K$ and EXIF-$K$ point maps.

In [ ]:
# Do not modify
def floor_frame(P_map, mask, n, d):
    P = P_map[mask]
    h = P @ n + d                                    # height above floor
    e1 = np.array([1.0, 0, 0]) - n[0] * n            # camera right, projected onto the floor
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(e1, n)                             # forward along the floor
    if e2[2] < 0: e2 = -e2
    keep = (h > 0.3) & (h < 2.0)
    return P[keep] @ e1, P[keep] @ e2

plt.figure(figsize=(8, 7))
for name, P_map, color in [("MapAnything K", pts_ma, "C0"), ("EXIF K", pts_exif, "C3")]:
    x, y = floor_frame(P_map, valid, *floor[name])
    plt.scatter(x, y, s=0.5, c=color, alpha=0.3)
    plt.scatter([], [], s=30, c=color, label=name)   # legend entry
plt.plot(0, 0, "k^", ms=10, label="camera")
plt.xlabel("right [m]"); plt.ylabel("forward [m]"); plt.title("Obstacles 0.3 - 2 m above the floor (top view)")
plt.gca().set_aspect("equal"); plt.legend(); plt.grid(alpha=0.3); plt.show()

**Question:** In the top view, how do the two point maps differ in where they place the poles? Do the poles spread apart or pinch together, and how does that follow from the sign of `focal_diff_pct`? If a drone used this map to fly between two poles, which would hurt more: a 5% error in depth, or a 5% error in focal length? Explain.

*Your answer here.*

### Measuring the poles 
Finally, measure two things in the room with each point map.

Use the zoomable plot below to pick pixel coordinates $(u, v)$ in the **518 $\times$ 392 network image** for:
- `base_A`, `base_B`: the center of the bases of the two poles you measured with the tape measure,
- `foam_top`, `foam_bottom`: the top and bottom ends of one blue foam tube.

Then fill in `dist3d`, which returns the 3D distance between two pixels in a given point map. To make it less sensitive to a single noisy pixel, use the **median** 3D point over a small $(2k+1)\times(2k+1)$ window around each pixel, ignoring invalid pixels.

In [ ]:
# Do not modify
plt.figure(figsize=(12, 9))
plt.imshow(img_net)
plt.xticks(range(0, W_net, 20), fontsize=7); plt.yticks(range(0, H_net, 20), fontsize=7)
plt.grid(alpha=0.4, color="r"); plt.title("Pick pixel coordinates (u = column, v = row)"); plt.show()

In [ ]:
# TODO: (u, v) pixel coordinates in the 518 x 392 image
base_A      = None
base_B      = None
foam_top    = None
foam_bottom = None

# TODO
def dist3d(P_map, mask, px1, px2, k=2):
    '''
    P_map: (H, W, 3) point map, mask: (H, W) boolean validity mask
    px1, px2: (u, v) pixel coordinates
    returns: Euclidean distance [m] between the median 3D points around px1 and px2
    '''
    pass

In [ ]:
# Do not modify
fig, ax = plt.subplots(figsize=(7, 5.3))
ax.imshow(img_net)
ax.plot(*zip(base_A, base_B), "yo-", lw=2, label="pole spacing")
ax.plot(*zip(foam_top, foam_bottom), "co-", lw=2, label="foam length")
ax.legend(); ax.axis("off"); plt.show()

print(f"{'':>15} | {'cam height':>10} | {'pole spacing':>12} | {'foam length':>11}")
print("-" * 58)
for name, P_map, m in [("MapAnything K", pts_ma, valid), ("EXIF K", pts_exif, valid), ("Calibrated run", pts_cal, valid_cal)]:
    print(f"{name:>15} | {abs(floor[name][1]):>8.3f} m | {dist3d(P_map, m, base_A, base_B):>10.3f} m | "
          f"{dist3d(P_map, m, foam_top, foam_bottom):>9.3f} m")
print(f"{'Tape measure':>15} | {CAMERA_HEIGHT_MEASURED!s:>8} m | {POLE_SPACING_MEASURED!s:>10} m | {FOAM_LENGTH_MEASURED!s:>9} m")

**Question:** Summarize the table. Which of the three point maps agrees best with the tape measure, and is the difference between them larger or smaller than the overall error relative to ground truth? Based on this lab, if you were building a map of the flight space for path planning, would you trust a single-image metric depth network, and what would you do to make the map more reliable? 

*Your answer here.*

# Submission Instructions

Please submit your completed notebook (with all outputs visible) and the photo you used to Gradescope.